# Get To Know A Dataset: Independent Copy of NASA PDS Geosciences Node Data

This draft notebook introduces a **proposed independent copy of public source data** from the [**NASA Planetary Data System (PDS) Geosciences Node**](https://pds-geosciences.wustl.edu/), proposed for the [Registry of Open Data on AWS](https://registry.opendata.aws/). The copy's operator is **WashU EEPS (Washington University in St. Louis Department of Earth, Environmental, and Planetary Sciences)**. This independent copy is **not operated or endorsed by NASA or PDS**.

The source Geosciences Node archive holds calibrated and derived data for the surfaces and interiors of terrestrial planetary bodies, including Mars, the Moon, Venus, and Mercury, from missions such as Mars Reconnaissance Orbiter, Mars Science Laboratory (Curiosity), Mars 2020 (Perseverance), Mars Odyssey, Mars Express, and Lunar Reconnaissance Orbiter. These examples describe the source archive, not confirmed coverage of the proposed copy. The copy's content and coverage are not fixed; no complete mirror is claimed.

| Resource | Value |
|----------|-------|
| **Operator of proposed copy** | WashU EEPS |
| **S3 bucket** | Unprovisioned; `pds-geosciences` is an unverified placeholder |
| **Region** | `us-west-2` is an unverified placeholder |
| **Access** | Proposed public, anonymous access (unsigned requests); not live |
| **Update schedule** | To be determined |
| **Source documentation** | [Geosciences Node](https://pds-geosciences.wustl.edu/) |

> **Draft only:** The bucket has not been provisioned. Confirm the published bucket name, region, and available content with WashU EEPS before running S3 cells; a response from a placeholder bucket does not verify its ownership or provenance. The examples include handling for missing or unavailable resources.


## Source formats and proposed copy layout

PDS archives are organized as **standards-compliant products** rather than ad hoc files. The proposed independent copy's content, coverage, and S3 layout are not fixed. Source products may include:

- **PDS3** label + image pairs (`.lbl` / `.img`, or detached labels)
- **PDS4** bundles with XML labels and binary or image extensions
- **Cloud-friendly rasters** where available (for example Cloud Optimized GeoTIFF)

No mission, product type, or prefix is guaranteed to be included. Once the operator publishes verified resource details, the code in the next sections can discover the structure present at run time.


In [ ]:
# Library requirements are listed in pds-geosciences/requirements.txt
# Install with: pip install -r pds-geosciences/requirements.txt

from __future__ import annotations

from typing import Any

import boto3
import matplotlib.pyplot as plt
import numpy as np
import s3fs
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError

# Unverified placeholders: confirm the operator-published bucket and region before running.
BUCKET = "pds-geosciences"
REGION = "us-west-2"

# Unsigned S3 client for public bucket access (AWS Open Data pattern)
s3 = boto3.client(
    "s3",
    region_name=REGION,
    config=Config(signature_version=UNSIGNED),
)

# s3fs provides a filesystem-like interface to the same bucket
fs = s3fs.S3FileSystem(anon=True, client_kwargs={"region_name": REGION})

print(f"Configured example: s3://{BUCKET}/ in {REGION} (anonymous; verify resource details before use)")



## Browse the bucket with boto3 and s3fs

We list top-level prefixes with `list_objects_v2` (boto3) and compare with `s3fs` globbing. If the bucket is missing or empty, you will see a short status message instead of a traceback.


In [ ]:
def bucket_status(s3_client, bucket: str) -> dict[str, Any]:
    """Return {'ok': bool, 'message': str, 'prefixes': list[str]}."""
    try:
        s3_client.head_bucket(Bucket=bucket)
    except ClientError as exc:
        code = exc.response.get("Error", {}).get("Code", "")
        if code in ("404", "NoSuchBucket", "NotFound"):
            return {
                "ok": False,
                "message": (
                    f"Bucket s3://{bucket}/ does not exist yet (or is not public in {REGION}). "
                    "The proposed copy is not provisioned; confirm the operator-published bucket and region "
                    "before re-running this notebook."
                ),
                "prefixes": [],
            }
        return {"ok": False, "message": f"head_bucket failed: {code} — {exc}", "prefixes": []}

    resp = s3_client.list_objects_v2(Bucket=bucket, Delimiter="/", MaxKeys=1000)
    prefixes = [p["Prefix"] for p in resp.get("CommonPrefixes", [])]
    if not prefixes and resp.get("KeyCount", 0) == 0:
        return {
            "ok": True,
            "message": f"Bucket s3://{bucket}/ exists but has no listable top-level prefixes yet (may be empty).",
            "prefixes": [],
        }
    return {
        "ok": True,
        "message": f"Found {len(prefixes)} top-level prefix(es) in s3://{bucket}/.",
        "prefixes": prefixes,
    }


status = bucket_status(s3, BUCKET)
print(status["message"])
if status["prefixes"]:
    print("\nTop-level prefixes (boto3):")
    for prefix in status["prefixes"][:20]:
        print(" ", prefix)
    if len(status["prefixes"]) > 20:
        print(f"  ... and {len(status['prefixes']) - 20} more")


In [ ]:
if status["ok"]:
    s3_path = f"s3://{BUCKET}/"
    try:
        # s3fs: list immediate children (non-recursive)
        entries = fs.ls(s3_path, detail=False)
        print(f"s3fs ls({s3_path}) returned {len(entries)} entries (showing up to 15):")
        for entry in entries[:15]:
            print(" ", entry)
    except FileNotFoundError:
        print("s3fs could not list the bucket root (bucket missing or no access).")
    except Exception as exc:  # noqa: BLE001 — tutorial notebook; surface any s3fs error clearly
        print(f"s3fs listing skipped: {type(exc).__name__}: {exc}")
else:
    print("Skipping s3fs browse because bucket_status reported the bucket is unavailable.")


In [ ]:
MISSION_HINTS = ("mars", "mro", "msl", "m2020", "lro", "moon", "lunar", "venus", "mercury", "odyssey", "mex")
MARS_MOON_HINTS = ("mars", "mro", "msl", "m2020", "lro", "moon", "lunar")
PLOT_EXTENSIONS = (".tif", ".tiff", ".jpg", ".jpeg", ".png", ".img")


def find_candidate_keys(s3_client, bucket: str, max_keys: int = 500) -> list[str]:
    """Return plottable keys, with Mars/Moon mission matches ranked before other missions and fallbacks."""
    if not status["ok"]:
        return []
    mars_moon: list[str] = []
    other_mission: list[str] = []
    fallback: list[str] = []
    paginator = s3_client.get_paginator("list_objects_v2")
    try:
        for page in paginator.paginate(Bucket=bucket, PaginationConfig={"MaxItems": max_keys}):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                lower = key.lower()
                if not lower.endswith(PLOT_EXTENSIONS):
                    continue
                if any(hint in lower for hint in MARS_MOON_HINTS):
                    mars_moon.append(key)
                elif any(hint in lower for hint in MISSION_HINTS):
                    other_mission.append(key)
                else:
                    fallback.append(key)
    except ClientError as exc:
        print(f"Object listing failed: {exc}")
        return []

    if mars_moon:
        return mars_moon + other_mission
    if other_mission:
        return other_mission
    return fallback


candidates = find_candidate_keys(s3, BUCKET)
print(f"Candidate product keys for demo plotting: {len(candidates)}")
for key in candidates[:10]:
    print(" ", key)



## Open representative products and plot

When data are available, we prefer products whose keys suggest **Mars** or **Moon/LRO** origin. **GeoTIFF/COG** products are opened with **rasterio** directly on `s3://` URLs (unsigned) and read via **overviews or a bounded `out_shape` preview**, so multi-gigabyte rasters are not loaded into memory. PNG/JPEG previews are thumbnail-capped; raw PDS3 `.IMG` files require labels and are rejected when too large to read whole-file.

Contrast stretching ignores **nodata** sentinels (for example `-32768`) declared on the dataset or carried in masked arrays. **RGB** composites are displayed in color rather than as a single band.



In [ ]:
MAX_PREVIEW_DIM = 1024
# Non-COG PNG/JPEG and raw PDS3 .IMG are capped; GeoTIFF uses rasterio overviews/windows instead.
MAX_WHOLE_FILE_BYTES = 32 * 1024 * 1024

RASTERIO_AWS_ENV = {
    "AWS_NO_SIGN_REQUEST": "YES",
    "AWS_REGION": REGION,
}


def s3_uri(bucket: str, key: str) -> str:
    return f"s3://{bucket}/{key}"


def object_size_bytes(s3_client, bucket: str, key: str) -> int | None:
    try:
        return int(s3_client.head_object(Bucket=bucket, Key=key)["ContentLength"])
    except ClientError:
        return None


def preview_out_shape(height: int, width: int, max_dim: int = MAX_PREVIEW_DIM) -> tuple[int, int]:
    scale = max(height, width) / max_dim
    if scale <= 1:
        return height, width
    out_h = max(1, int(round(height / scale)))
    out_w = max(1, int(round(width / scale)))
    return out_h, out_w


def valid_for_stretch(values: np.ndarray, nodata: float | int | None) -> np.ndarray:
    """Boolean mask of pixels to include in contrast stretch (excludes nodata and non-finite)."""
    if np.ma.isMaskedArray(values):
        mask = ~np.ma.getmaskarray(values)
        data = values.data
    else:
        mask = np.ones(values.shape, dtype=bool)
        data = values
    if nodata is not None:
        try:
            mask &= data != nodata
        except TypeError:
            pass
    if np.issubdtype(data.dtype, np.floating):
        mask &= np.isfinite(data)
    return mask


def percentile_stretch_limits(values: np.ndarray, nodata: float | int | None) -> tuple[float | None, float | None]:
    mask = valid_for_stretch(values, nodata)
    if not mask.any():
        return None, None
    sample = np.asarray(values)[mask].astype(np.float64)
    return tuple(np.percentile(sample, [2, 98]))  # type: ignore[return-value]


def load_image_array_from_s3(
    s3_client,
    bucket: str,
    key: str,
    filesystem: s3fs.S3FileSystem,
) -> tuple[np.ndarray | None, str, float | int | None]:
    """Load a downsampled preview without downloading multi-GB products whole-file."""
    lower = key.lower()
    uri = s3_uri(bucket, key)

    if lower.endswith((".tif", ".tiff")):
        try:
            import rasterio
            from rasterio.enums import Resampling

            with rasterio.Env(**RASTERIO_AWS_ENV):
                with rasterio.open(uri) as dataset:
                    nodata = dataset.nodata
                    out_h, out_w = preview_out_shape(dataset.height, dataset.width)
                    out_shape = (out_h, out_w)
                    read_kwargs: dict[str, Any] = {
                        "out_shape": out_shape,
                        "resampling": Resampling.average,
                        "masked": True,
                    }
                    if dataset.overviews(1):
                        msg = (
                            f"Loaded GeoTIFF preview via overviews/out_shape {out_w}x{out_h} "
                            f"(full size {dataset.width}x{dataset.height})"
                        )
                    else:
                        msg = (
                            f"Loaded GeoTIFF windowed preview {out_w}x{out_h} "
                            f"(full size {dataset.width}x{dataset.height})"
                        )
                    if dataset.count >= 3:
                        bands = dataset.read([1, 2, 3], **read_kwargs)
                        rgb = np.ma.stack(bands, axis=-1)
                        return rgb, msg + " RGB", nodata
                    band = dataset.read(1, **read_kwargs)
                    return band, msg, nodata
        except Exception as exc:  # noqa: BLE001
            return None, f"rasterio could not read {key}: {exc}", None

    if lower.endswith((".png", ".jpg", ".jpeg")):
        size = object_size_bytes(s3_client, bucket, key)
        if size is not None and size > MAX_WHOLE_FILE_BYTES:
            return (
                None,
                f"Skipping {key}: {size / 1e6:.1f} MB exceeds preview cap; prefer COG GeoTIFF with overviews.",
                None,
            )
        try:
            from PIL import Image

            with filesystem.open(uri, "rb") as handle:
                image = Image.open(handle)
                image.thumbnail((MAX_PREVIEW_DIM, MAX_PREVIEW_DIM))
                array = np.asarray(image)
            return array, f"Loaded thumbnail with Pillow ({image.mode})", None
        except Exception as exc:  # noqa: BLE001
            return None, f"Pillow could not read {key}: {exc}", None

    if lower.endswith(".img"):
        size = object_size_bytes(s3_client, bucket, key)
        if size is None:
            return None, f"Could not determine size for {key}", None
        if size > MAX_WHOLE_FILE_BYTES:
            return (
                None,
                "PDS3 .IMG is too large to read whole-file; use the paired .LBL with GDAL/rasterio windowed reads.",
                None,
            )
        try:
            with filesystem.open(uri, "rb") as handle:
                body = handle.read()
        except Exception as exc:  # noqa: BLE001
            return None, f"Could not read {key}: {exc}", None
        side = int(np.sqrt(len(body)))
        if side > 0 and side * side == len(body):
            arr = np.frombuffer(body, dtype=np.uint8).reshape((side, side))
            return arr, f"Interpreted .IMG as {side}x{side} uint8 (verify with PDS label)", None
        return (
            None,
            "PDS3 .IMG usually needs its .LBL label (dimensions, dtype, band storage). "
            "Use GDAL PDS driver with range/window reads.",
            None,
        )

    return None, f"No loader implemented for extension of {key}", None


def plot_product(s3_client, bucket: str, key: str, filesystem: s3fs.S3FileSystem) -> None:
    data, msg, nodata = load_image_array_from_s3(s3_client, bucket, key, filesystem)
    print(msg)
    if data is None:
        return

    plt.figure(figsize=(8, 8))

    # RGB/RGBA composites (PNG/JPEG or multi-band GeoTIFF)
    if data.ndim == 3 and data.shape[-1] >= 3:
        rgb = np.ma.asarray(data[..., :3])
        display = np.zeros(rgb.shape, dtype=np.float64)
        for channel in range(3):
            band = rgb[..., channel]
            vmin, vmax = percentile_stretch_limits(band, nodata)
            if vmin is None or vmax is None or vmax <= vmin:
                display[..., channel] = np.ma.filled(band.astype(np.float64), 0.0)
            else:
                scaled = (band.astype(np.float64) - vmin) / (vmax - vmin)
                display[..., channel] = np.clip(np.ma.filled(scaled, 0.0), 0.0, 1.0)
        plt.imshow(display)
    else:
        band = np.ma.asarray(data)
        if band.ndim == 3:
            band = band[..., 0]
        vmin, vmax = percentile_stretch_limits(band, nodata)
        plt.imshow(band, cmap="gray", vmin=vmin, vmax=vmax)

    plt.title(key.split("/")[-1][:80])
    plt.axis("off")
    plt.tight_layout()
    plt.show()


if not status["ok"]:
    print("Bucket unavailable — skipping product plots. Confirm the operator-published resource details before use.")
elif not candidates:
    print(
        "No plottable candidate objects found in the first scan. "
        "After data upload, re-run find_candidate_keys() or set SAMPLE_KEYS manually."
    )
    SAMPLE_KEYS: list[str] = []
else:
    SAMPLE_KEYS = candidates[:2]
    print("Plotting up to two sample products:")
    for sample_key in SAMPLE_KEYS:
        print("\n---", sample_key, "---")
        plot_product(s3, BUCKET, sample_key, fs)



## Citation and next steps

Attribute the original source data to NASA PDS and the relevant data producers, and cite the specific PDS datasets using the guidance on the [Geosciences Node FAQ](https://pds-geosciences.wustl.edu/faq.html). For large-scale analysis, combine anonymous S3 access with tools such as [STAC catalogs](https://stacspec.org/), [xarray](https://xarray.dev/), or [Dask](https://www.dask.org/) over `s3fs` if suitable cloud-native formats (for example COG or Zarr) are included in the independent copy. Its format availability and coverage are not yet fixed. Source attribution does not imply NASA/PDS operation or endorsement of this copy.

**Useful links**

- [PDS Geosciences Node](https://pds-geosciences.wustl.edu/)
- [PDS standards documentation](https://pds.nasa.gov/datastandards/documentation/)
- [AWS Open Data Registry](https://registry.opendata.aws/) (proposed listing; publication and resource details are not yet confirmed)


## Community challenge

**Question:** *How does surface texture (roughness at 10–100 m scales) differ between ancient Noachian highlands and younger Amazonian volcanic plains on Mars, using orbital image products you stream from S3 without downloading the full archive?*

**Why it matters:** Geomorphological province boundaries drive landing-site safety, resource mapping, and climate history studies. Cloud access to PDS orbital data makes regional comparisons feasible on a laptop if you only pull the tiles you need.

**Suggested approach**

1. If suitable MRO / Odyssey / MEX image or DEM products are included, discover them with the browse helpers above after confirming the operator-published bucket and region. These products are not guaranteed to be included.
2. Build a small AOI (area of interest) over one highland region and one plain region.
3. Compute a texture metric (for example variance of Laplacian, or GLCM contrast) on co-registered subsets.
4. Share a notebook that reads **only** the bytes required via range requests, and document citation metadata for each PDS product ID.

We would love to see your notebook linked from a GitHub issue or pull request in [this examples repository](https://github.com/fossettlab/open-data-examples).
